# CID 4 Taxonomy Text Baseline

This notebook is the notebook half of the mixed deliverable. It reuses the shared `ml.datasets` package instead of duplicating feature engineering logic from scratch.

The workflow is intentionally lightweight: it uses the taxonomy dataset already present in the repository, builds a text baseline over organism and taxonomy strings, and saves reusable artifacts for later comparison with the tabular and future deep-learning workflows.

In [1]:
import json
import pickle
import re
import sys
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

project_root = Path.cwd().resolve().parent if Path.cwd().name == "src" else Path.cwd().resolve()
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from ml.datasets import build_taxonomy_clustering_frame  # noqa: E402

## 1. Load and Inspect the Data

Load the taxonomy table through the shared dataset builder, inspect the columns, and verify which rows are suitable for a small text baseline.

In [2]:
taxonomy_df = build_taxonomy_clustering_frame()
taxonomy_df["text"] = (
    taxonomy_df["Source_Organism"].astype(str).str.strip() + " " + taxonomy_df["Taxonomy"].astype(str).str.strip()
)

print("Shape:", taxonomy_df.shape)
display(taxonomy_df.head())
display(taxonomy_df.dtypes.to_frame("dtype"))
display(taxonomy_df[["Source_Organism", "Taxonomy", "animalClass", "text"]].isna().sum().to_frame("missing"))

Shape: (29, 5)


,Source_Organism,Taxonomy,Taxonomy_ID,animalClass,text
0,Gallus gallus,Gallus gallus (chicken),9031,bird,Gallus gallus Gallus gallus (chicken)
1,Dromaius novaehollandiae,Dromaius novaehollandiae (emu),8790,bird,Dromaius novaehollandiae Dromaius novaeholland...
2,Struthio camelus,Struthio camelus (African ostrich),8801,bird,Struthio camelus Struthio camelus (African ost...
3,Ovis aries,Ovis aries (sheep),9940,mammal,Ovis aries Ovis aries (sheep)
4,Bos taurus,Bos taurus (domestic cattle),9913,mammal,Bos taurus Bos taurus (domestic cattle)


,dtype
Source_Organism,object
Taxonomy,object
Taxonomy_ID,int64
animalClass,object
text,object


,missing
Source_Organism,0
Taxonomy,0
animalClass,0
text,0


## 2. Clean and Normalize Text

Standardize the text fields before vectorization so the baseline uses a consistent representation.

In [3]:
def normalize_text(value: str) -> str:
    lowered = str(value).lower().strip()
    lowered = re.sub(r"[^a-z0-9\s]", " ", lowered)
    lowered = re.sub(r"\s+", " ", lowered)
    return lowered


model_df = taxonomy_df.loc[taxonomy_df["animalClass"].isin(["bird", "mammal"])].copy()
model_df["clean_text"] = model_df["text"].map(normalize_text)
model_df = model_df.loc[model_df["clean_text"].ne("")].reset_index(drop=True)

print("Retained rows:", len(model_df))
display(model_df[["Source_Organism", "animalClass", "clean_text"]].head())

Retained rows: 29


,Source_Organism,animalClass,clean_text
0,Gallus gallus,bird,gallus gallus gallus gallus chicken
1,Dromaius novaehollandiae,bird,dromaius novaehollandiae dromaius novaeholland...
2,Struthio camelus,bird,struthio camelus struthio camelus african ostr...
3,Ovis aries,mammal,ovis aries ovis aries sheep
4,Bos taurus,mammal,bos taurus bos taurus domestic cattle


## 3. Tokenize and Vectorize Inputs

Convert the cleaned text into model-ready features with TF-IDF. This keeps the notebook simple while still giving a meaningful baseline over the taxonomy strings.

In [4]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
feature_matrix = vectorizer.fit_transform(model_df["clean_text"])

print("Vectorized shape:", feature_matrix.shape)
pd.Series(vectorizer.get_feature_names_out()).head(20).to_frame("feature")

Vectorized shape: (29, 171)


,feature
0,aegagrus
1,aegagrus hircus
2,african
3,african ostrich
4,american
5,american bison
6,anas
7,anas platyrhynchos
8,anatidae
9,anatidae anatidae


## 4. Build a Baseline Model

Split the retained rows, train a simple text classifier, and generate initial predictions.

In [5]:
x_train, x_test, y_train, y_test = train_test_split(
    model_df["clean_text"],
    model_df["animalClass"],
    test_size=0.3,
    random_state=42,
    stratify=model_df["animalClass"],
)

text_pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)
text_pipeline.fit(x_train, y_train)
predictions = text_pipeline.predict(x_test)

pd.DataFrame({"text": x_test.tolist(), "actual": y_test.tolist(), "predicted": predictions.tolist()}).head()

,text,actual,predicted
0,anas platyrhynchos anas platyrhynchos mallard,bird,mammal
1,bos taurus x bison bison bos taurus x bison bi...,mammal,mammal
2,sus scrofa domestica sus scrofa domesticus dom...,mammal,mammal
3,sus scrofa sus scrofa pig,mammal,mammal
4,anatidae anatidae waterfowl,bird,mammal


## 5. Evaluate Model Performance

Compute a compact metric summary and inspect the confusion matrix so weak spots are obvious.

In [6]:
report = classification_report(y_test, predictions, output_dict=True, zero_division=0)
confusion = confusion_matrix(y_test, predictions, labels=["bird", "mammal"])

print(json.dumps(report, indent=2))
pd.DataFrame(confusion, index=["actual_bird", "actual_mammal"], columns=["pred_bird", "pred_mammal"])

{
  "bird": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 4.0
  },
  "mammal": {
    "precision": 0.5555555555555556,
    "recall": 1.0,
    "f1-score": 0.7142857142857143,
    "support": 5.0
  },
  "accuracy": 0.5555555555555556,
  "macro avg": {
    "precision": 0.2777777777777778,
    "recall": 0.5,
    "f1-score": 0.35714285714285715,
    "support": 9.0
  },
  "weighted avg": {
    "precision": 0.30864197530864196,
    "recall": 0.5555555555555556,
    "f1-score": 0.39682539682539686,
    "support": 9.0
  }
}


,pred_bird,pred_mammal
actual_bird,0,4
actual_mammal,0,5


## 6. Run Predictions on New Samples

Pass a few new organism strings through the same preprocessing and model pipeline.

In [7]:
new_samples = pd.Series(
    [
        normalize_text("Gallus gallus domesticus chicken"),
        normalize_text("Bos taurus cattle"),
        normalize_text("Meleagris gallopavo turkey"),
        normalize_text("Capra hircus goat"),
    ]
)

pd.DataFrame({"sample": new_samples, "prediction": text_pipeline.predict(new_samples)})

,sample,prediction
0,gallus gallus domesticus chicken,bird
1,bos taurus cattle,mammal
2,meleagris gallopavo turkey,bird
3,capra hircus goat,mammal


## 7. Save Processed Artifacts

Persist the fitted pipeline and a compact metadata summary so the baseline can be reused outside the notebook.

In [8]:
artifact_dir = project_root / ".." / "data" / "out"
artifact_dir = artifact_dir.resolve()
artifact_dir.mkdir(parents=True, exist_ok=True)

with (artifact_dir / "cid4_ml.taxonomy_text_pipeline.pkl").open("wb") as handle:
    pickle.dump(text_pipeline, handle)

metadata = {
    "row_count": int(len(model_df)),
    "label_counts": model_df["animalClass"].value_counts().to_dict(),
    "feature_count": int(len(text_pipeline.named_steps["tfidf"].get_feature_names_out())),
}
with (artifact_dir / "cid4_ml.taxonomy_text_pipeline.summary.json").open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

metadata

{'row_count': 29,
 'label_counts': {'mammal': 15, 'bird': 14},
 'feature_count': 124}

## Shared ML Outputs and Current Limitations

If the main runner has already been executed, inspect the generated JSON summaries here.

Current limitations:
- the repository snapshot is still centered on a single molecule, so deep models remain scaffold-only
- PyTorch and TensorFlow may be absent from the current environment
- the filtered bioactivity Active vs Inactive slice may collapse to a single observed class, which makes several classifiers inapplicable

In [12]:
summary_paths = {
    "cross_library": artifact_dir / "cid4_ml.cross_library_comparison.summary.json",
    "sklearn_suite": artifact_dir / "cid4_ml.sklearn_suite.summary.json",
    "future_scaffolds": artifact_dir / "cid4_ml.future_scaffolds.summary.json",
}

loaded_summaries = {}
for name, path in summary_paths.items():
    if path.exists():
        with path.open("r", encoding="utf-8") as handle:
            loaded_summaries[name] = json.load(handle)

loaded_summaries.keys()

dict_keys(['cross_library', 'sklearn_suite', 'future_scaffolds'])

In [13]:
if loaded_summaries:
    scaffold_summary = loaded_summaries.get("future_scaffolds", {})
    display(pd.Series(scaffold_summary.get("gnn", {}), name="gnn_scaffold"))
    display(pd.Series(scaffold_summary.get("smiles_rnn", {}), name="smiles_rnn_scaffold"))
    display(pd.Series(loaded_summaries.get("cross_library", {}).keys(), name="cross_library_tasks"))
else:
    print("Run python src/cid4_ml.py from the py workspace first to populate data/out summaries.")

status                                                              scaffold_only
model_family                                                 graph-neural-network
dataset                         {'name': 'atom-heavy-vs-hydrogen', 'task_type'...
recommended_primary_library                                     pytorch-geometric
recommended_alternatives                                          [dgl, deepchem]
why_not_fully_implemented       The current CID 4 snapshot is effectively a si...
current_reusable_inputs         {'atom_features': ['bondCount', 'totalHydrogen...
minimum_dataset_requirements    [A corpus of many molecules instead of a singl...
proposed_pipeline               [Use RDKit to convert each molecule into atom ...
next_code_targets               [Add a graph dataset builder that returns Data...
Name: gnn_scaffold, dtype: object

status                                                              scaffold_only
model_family                                                           smiles-rnn
recommended_primary_library                                            tensorflow
recommended_alternatives                           [pytorch, keras-nlp, deepchem]
why_not_fully_implemented       The repository currently centers on one molecu...
minimum_dataset_requirements    [A larger corpus of unique SMILES strings., A ...
proposed_pipeline               [Canonicalize SMILES strings with RDKit., Toke...
starter_architecture            {'embedding_dim': 64, 'recurrent_layer': 'LSTM...
next_code_targets               [Add a SMILES corpus loader from a multi-molec...
Name: smiles_rnn_scaffold, dtype: object

0               atom_heavy_vs_hydrogen
1              atom_element_multiclass
2    bioactivity_binary_classification
3            activity_value_regression
Name: cross_library_tasks, dtype: object